## To visually review dataset column headers, map data-cleaning steps, and evaluate performance changes easily.

##### What this notebook for:

It interacts strictly with the internal content of the files using your active notebook kernel.

##### Its Core Job:

It uses SageMaker Fast File Mode to stream rows out of your S3 bucket on demand without downloading them to disk. It handles string column cleaning (lowercasing, stripping weird spaces), executes the unsupervised TF-IDF character n-gram blocking logic, and drops intermediate duplicate records to safeguard your pipeline against parsing bugs.

##### Frequency:

We will run, tweak, and re-execute this notebook constantly during your development phase to refine your text processing and maximize your high-recall candidate pairings before exporting it to your final submission folder.

In [ ]:
import os
import boto3
import pandas as pd

# 1. Establish structural path variables matching your AWS environment configuration
s3_bucket = "amazon-ml-challenge-2026-s3-bucket"   # <── Change this to your s3-bucket name
print(f"-> Verifying data stream paths for target bucket: {s3_bucket}")

s3_client = boto3.client('s3')
try:
    # Quick metadata lookup check to confirm bucket visibility
    s3_client.head_bucket(Bucket=s3_bucket)
    print("\n ✅ Success! S3 Storage infrastructure is initialized and fully reachable.")
except Exception as e:
    print("\n Connection Error: Ensure bucket naming configurations match your profile exactly.", e)

In [ ]:
# import io
# import gc

# def clean_text_columns(series):
#     """Standardizes row text alignments to prevent string formatting mismatch bugs."""
#     return series.fillna("").astype(str).str.lower().str.strip()

# print("-> Initializing stream ingestion for Reference Source 1 tables...")

# # 1. Fetch object components directly from the S3 prefix path bounds
# s1_obj = s3_client.get_object(Bucket=s3_bucket, Key="raw-data/train/train_source1.tsv")

# # 2. Ingest the data frame in a memory-safe stream block
# df_s1 = pd.read_csv(io.BytesIO(s1_obj['Body'].read()), sep="\t")

# # DEDUPLICATION CHECK: Drop repeated entry blocks to protect pipeline throughput
# df_s1 = df_s1.drop_duplicates(subset=["entity_id"])

# # Sanitize primary target columns
# df_s1["business_name"] = clean_text_columns(df_s1["business_name"])
# df_s1["business_address"] = clean_text_columns(df_s1["business_address"])

# print(f"\n ✅ Ingestion complete. Staged {len(df_s1)} clean unique entities for evaluation.\n")
# # Visual audit check: Print the top 5 clean tracking frames
# df_s1.head()

To stop memory leak in above Cell completely, we must stop using s1_obj['Body'].read().

That command forces your system to load the entire raw text object into RAM at once, causing that huge jump.Instead, we will use Boto3 streaming body chunking. This streams raw data chunks from S3 directly into memory-safe Pandas partitions, keeping our memory flat and stable.

In [ ]:
import io
import gc
import pandas as pd
from tqdm.notebook import tqdm

def clean_text_columns(series):
    """Standardizes row text alignments to prevent string formatting mismatch bugs."""
    return series.fillna("").astype(str).str.lower().str.strip()

print("-> Initializing memory-safe streaming ingestion for Reference Source 1...")

# 1. Establish data stream handles directly from the S3 bucket object body
s1_obj = s3_client.get_object(Bucket=s3_bucket, Key="raw-data/train/train_source1.tsv")
total_bytes = s1_obj['ContentLength']

# 2. Stream chunk-by-chunk using a byte buffer pool to keep memory low
chunks = []
chunk_size_bytes = 1024 * 1024 * 5  # Safe 5MB network streaming buffers

with tqdm(total=total_bytes, desc="Streaming Source 1", unit="B", unit_scale=True) as pbar:
    # Read the body in raw chunks instead of all at once
    for chunk_bytes in s1_obj['Body'].iter_chunks(chunk_size=chunk_size_bytes):
        chunks.append(chunk_bytes)
        pbar.update(len(chunk_bytes))

# Assemble byte stream arrays into a unified pandas structure cleanly
raw_bytes_stream = b"".join(chunks)
df_s1 = pd.read_csv(io.BytesIO(raw_bytes_stream), sep="\t")

# 3. Aggressive Deduplication & Text Sanitation Gateways
df_s1 = df_s1.drop_duplicates(subset=["entity_id"])
df_s1["business_name"] = clean_text_columns(df_s1["business_name"])
df_s1["business_address"] = clean_text_columns(df_s1["business_address"])

print("\n" + "="*66)
print(f"* Ingestion Complete! \n")
print(f"-> Staged Records Count: {len(df_s1)}")
print("="*66 + "\n")

# Free up temporary structural parameters out of memory caches instantly
del chunks, raw_bytes_stream
gc.collect()

# Display the top 5 clean tracking frames visually
df_s1.head()


In [ ]:
# import gc
# from sklearn.feature_extraction.text import TfidfVectorizer

# print("-> Compiling dynamic character n-gram tokens safely...")

# # 1. Avoid converting to a massive Python list (tolist() duplicates data in RAM!)
# # Instead, feed the Pandas Series directly into the vectorizer as a raw array string mapping
# name_corpus = df_s1["business_name"].values

# # 2. Optimized parameter constraints to drastically prevent memory spikes:
# # - sublinear_tf=True scales down massive term frequencies efficiently
# # - min_df=2 ignores rare 1-off noise character combinations completely
# vectorizer = TfidfVectorizer(
#     analyzer="char_wb", 
#     ngram_range=(3, 5), 
#     max_features=25000,
#     min_df=2,
#     sublinear_tf=True
# )

# # Run the fit operation
# s1_sparse_matrices = vectorizer.fit_transform(name_corpus)

# # Note: Using .shape[1] prints the correct number of feature columns
# print(f"✅ Dynamic vocabulary generation finalized. Tracked {s1_sparse_matrices.shape[1]} unique character tokens.")
# print("-> Blocker workspace initialized. Ready to export high-recall candidate profiles.")

# # 3. Aggressive Memory Release Execution
# del name_corpus
# gc.collect()

Instead of creating a memory-hungry vocabulary dictionary in memory, implementing stream-batched and a hashing vectorizer will use a mathematical hash function to map characters directly to fixed coordinate dimensions. It requires almost zero extra RAM, runs at faster speeds, and our instance will not lock up.

In [ ]:
# import gc
# import scipy.sparse as sp
# from sklearn.feature_extraction.text import HashingVectorizer, TfidfTransformer
# from sklearn.pipeline import make_pipeline
# from tqdm.notebook import tqdm

# print("-> Initializing memory-safe streaming Hashing configuration...")

# name_corpus = df_s1["business_name"].values
# total_records = len(name_corpus)
# vectorizer_batch_size = 10000  # Small, safe chunk size to keep memory flat

# # 1. Establish the lightweight vectorization layer
# hash_pipe = make_pipeline(
#     HashingVectorizer(
#         analyzer="char_wb",
#         ngram_range=(3, 5),
#         n_features=25000,
#         alternate_sign=False  # Keeps indices positive for Cosine Similarity evaluation
#     ),
#     TfidfTransformer(sublinear_tf=True)
# )

# print(f"-> Processing {total_records} records into sparse layouts...")
# batch_matrices = []

# # 2. Track the extraction loop using an explicit live visual progress bar
# with tqdm(total=total_records, desc="⚡ Hashing N-Grams", unit="rows") as pbar:
#     for i in range(0, total_records, vectorizer_batch_size):
#         chunk = name_corpus[i : i + vectorizer_batch_size]
        
#         # Transform the current tiny slice independently
#         chunk_sparse = hash_pipe.fit_transform(chunk)
#         batch_matrices.append(chunk_sparse)
        
#         pbar.update(len(chunk))
        
#         # Micro-cleanse memory variables at the end of each block step
#         del chunk, chunk_sparse
#         gc.collect()

# print("-> Merging processed matrices into a single coordinate layout...")
# # Stack the clean batches vertically into one unified compressed row matrix
# s1_sparse_matrices = sp.vstack(batch_matrices, format="csr")

# print("="*66)
# print(f"* Hashing Matrix Staging Finalized!")
# print(f"-> Sparse Target Shape: {s1_sparse_matrices.shape}")
# print("="*66)

# # 3. Final Memory Self-Destruct Strategy
# del batch_matrices, name_corpus
# gc.collect()

fit_transform on the TfidfTransformer re-calculates corpus-wide statistical states for every batch chunk.

To resolve this issue, we must split the processing steps. By using HashingVectorizer to transform the text blocks in pure streaming mode, we bypass vocabulary allocations completely.

## Pure-Streaming Matrix Ingestion

In [ ]:
import gc
import scipy.sparse as sp
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from tqdm.notebook import tqdm

print("-> Initializing pure streaming token hashing layout...")

name_corpus = df_s1["business_name"].values
total_records = len(name_corpus)
vectorizer_batch_size = 5000  # Smaller, safer slice window to keep memory flat

# 1. Initialize ONLY the Hashing layer (Bypasses all vocabulary dictionary states)
hasher = HashingVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    n_features=25000,
    alternate_sign=False
)

batch_matrices = []

# 2. Extract sparse coordinate features block-by-block
with tqdm(total=total_records, desc="⚡ Hashing N-Grams", unit="rows") as pbar:
    for i in range(0, total_records, vectorizer_batch_size):
        chunk = name_corpus[i : i + vectorizer_batch_size]
        
        # Transform the current block in pure isolation
        chunk_sparse = hasher.transform(chunk)
        batch_matrices.append(chunk_sparse)
        
        pbar.update(len(chunk))
        del chunk, chunk_sparse
        gc.collect()

print("-> Vertically stacking array structures...")
raw_sparse_matrix = sp.vstack(batch_matrices, format="csr")

# Clear out raw list array dependencies instantly from RAM
del batch_matrices, name_corpus
gc.collect()

print("-> Running low-overhead dynamic TF-IDF scaling transforms...")
# 3. Fit scaling metrics over the aggregated sparse matrix layout
transformer = TfidfTransformer(sublinear_tf=True)
s1_sparse_matrices = transformer.fit_transform(raw_sparse_matrix)

print("="*66)
print(f"* Hashing Matrix Staging Finalized!")
print(f"-> Sparse Target Dimensions: {s1_sparse_matrices.shape}")
print("="*66)

del raw_sparse_matrix
gc.collect()

In [ ]:
# import numpy as np
# from sklearn.metrics.pairwise import cosine_similarity

# print("-> Ingesting Target Sources 2 and 3 streams for candidate evaluation...")

# # 1. Stream the secondary dataset tables using strict tab separators
# s2_obj = s3_client.get_object(Bucket=s3_bucket, Key="raw-data/train/train_source2.tsv")
# s3_obj = s3_client.get_object(Bucket=s3_bucket, Key="raw-data/train/train_source3.tsv")

# df_s2 = pd.read_csv(io.BytesIO(s2_obj['Body'].read()), sep="\t").drop_duplicates(subset=["entity_id"])
# df_s3 = pd.read_csv(io.BytesIO(s3_obj['Body'].read()), sep="\t").drop_duplicates(subset=["entity_id"])

# # Combine target blocks into a single lookup index space
# df_s23 = pd.concat([df_s2, df_s3], ignore_index=True)

# print("-> Transforming text targets into sparse character vector dimensions...")
# s23_sparse_matrices = vectorizer.transform(clean_text_columns(df_s23["business_name"]))

# print("-> Executing block calculations (Letting a good volume of data pass)...")
# candidate_records = []

# # Compute matrix operations across vectors
# sim_matrix = cosine_similarity(s1_sparse_matrices, s23_sparse_matrices)

# for idx, s1_id in enumerate(df_s1["entity_id"]):
#     # Retain candidate targets displaying active similarity overlaps (> 0.15)
#     matching_indices = (sim_matrix[idx] > 0.15).nonzero()[0]
#     candidate_ids = df_s23.iloc[matching_indices]["entity_id"].tolist()
    
#     # Save elements out as a comma-separated string mapping index
#     candidate_records.append({
#         "source1_entity_id": s1_id,
#         "candidate_entity_ids": ",".join(candidate_ids)
#     })

# # 2. Export candidate targets to your persistent folder workspace for Step 3 ingestion
# df_candidates = pd.DataFrame(candidate_records)
# output_path = "../Package/output/candidate_pairs.tsv"
# df_candidates.to_csv(output_path, sep="\t", index=False)

# print(f"✅ Blocker step finalized! Exported {len(df_candidates)} space-reduced frames to: {output_path}")

# # Pipeline Memory Self-Destruct Strategy
# del df_s2, df_s3, df_s23, sim_matrix, candidate_records
# gc.collect()

In [ ]:
import gc
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.notebook import tqdm 

print("-> Initializing memory-safe batched blocker operations...")

s23_names = df_s23["business_name"].values
s1_ids = df_s1["entity_id"].values
s23_ids = df_s23["entity_id"].values

# 1. Transform the target datasets into space-saving sparse matrices
s23_sparse = vectorizer.transform(s23_names)

candidate_records = []
batch_size = 5000  # Safe boundary slice to prevent instance RAM spikes
total_rows = len(df_s1)

print(f"-> Processing {total_rows} reference rows in steps of {batch_size}...")

# This creates a fixed, active metrics board right below the running cell block
with tqdm(total=total_rows, desc= Blocker Progress", unit="rows") as pbar:
    for start_idx in range(0, total_rows, batch_size):
        end_idx = min(start_idx + batch_size, total_rows)
        
        # Slice the active coordinate matrix segment
        s1_batch_sparse = s1_sparse_matrices[start_idx:end_idx]
        
        # Compute optimized similarity dot products
        sim_matrix_batch = cosine_similarity(s1_batch_sparse, s23_sparse)
        
        # Extract matching profile coordinates from the active slice
        for local_idx in range(sim_matrix_batch.shape[0]):
            global_s1_idx = start_idx + local_idx
            s1_id = s1_ids[global_s1_idx]
            
            # Isolate matrix array hits clearing the threshold boundary
            matching_indices = (sim_matrix_batch[local_idx] > 0.15).nonzero()[0]
            candidate_ids = s23_ids[matching_indices]
            
            candidate_records.append({
                "source1_entity_id": s1_id,
                "candidate_entity_ids": ",".join(candidate_ids)
            })
            
        # Update the live metrics board with the chunk size delta value
        pbar.update(end_idx - start_idx)
        
        # Memory self-destruct cycle to protect instance RAM constraints
        del sim_matrix_batch
        gc.collect()

# 3. Export clean results natively to persistent output file structure
df_candidates = pd.DataFrame(candidate_records)
output_path = "../Package/output/candidate_pairs.tsv"
df_candidates.to_csv(output_path, sep="\t", index=False)

print("\n" + "="*66)
print(f"* Lossless Blocker Complete!")
print(f"-> Exported Entities Rows: {len(df_candidates)}")
print(f"-> Target Staging File:     {output_path} \n")
print("="*66)

# Clear structural parameters out of memory cache blocks
del candidate_records, s23_names
gc.collect()